In [76]:
from dataclasses import dataclass
import numpy as np
from tinygrad import Tensor, nn, dtypes
from tinygrad.nn.optim import AdamW
from tinygrad.nn.state import get_parameters
from typing import Optional
import math


In [77]:
@dataclass
class GPT2Config:
    vocab_size: int = 5              # GPT-2 byte-level BPE
    d_model: int = 8
    n_heads: int = 1
    n_layers: int = 1
    d_ff: int = 4                      # 4 * d_model
    max_seq_len: int = 4
    dropout: float = 0.1
    pad_idx: int = 0
    batch_size: int = 2

    @property
    def d_k(self) -> int:
        assert self.d_model % self.n_heads == 0
        return self.d_model // self.n_heads
    
cfg = GPT2Config(
        vocab_size=100,
        d_model=8, n_heads=2, n_layers=2, d_ff=6,
        max_seq_len=6, dropout=0.0, batch_size=2
    )


In [78]:
def make_causal_mask(size: int) -> Tensor:
    """Build an additive causal mask of shape (1, 1, T, T):
    -inf above the diagonal, 0 on and below. Broadcasts over (B, n_heads, T, T)
    when added to attention scores.
    """
    i = Tensor.arange(size).reshape(size, 1)
    j = Tensor.arange(size).reshape(1, size)
    future = j > i
    mask = future.where(Tensor(-float("inf")), Tensor(0.0))
    return mask.reshape(1, 1, size, size)

In [79]:
class ToyLMDataset:
    """Random next-token task for the smoke test.

    Yields (tokens_in, tokens_out, causal_mask) tuples. tokens_out is
    tokens_in shifted by one; ids start at 1 so pad_idx=0 is never produced.
    """
    def __init__(self, vocab_size: int, seq_len: int, n_batches: int, batch_size: int):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.n_batches = n_batches
        self.batch_size = batch_size

    def __iter__(self):
        for _ in range(self.n_batches):
            data = np.random.randint(1, self.vocab_size, (self.batch_size, self.seq_len + 1))
            tokens_in  = Tensor(data[:, :-1], dtype=dtypes.int32)
            tokens_out = Tensor(data[:, 1:],  dtype=dtypes.int32)
            causal_mask = make_causal_mask(self.seq_len)
            yield tokens_in, tokens_out, causal_mask



In [80]:
class TokenEmbedding:
    """Plain embedding matrix, no sqrt(d_model) scaling.

    Weight shape: (vocab_size, d_model).

    __call__:
        x:      (B, T)            int token ids in [0, vocab_size)
        return: (B, T, d_model)   float
    """
    def __init__(self, vocab_size: int, d_model: int):
        self.embedding = nn.Embedding(vocab_size, d_model)

    def __call__(self, x: Tensor) -> Tensor:
        # x: (B, T) int -> (B, T, d_model)
        return self.embedding(x)

    def __repr__(self):
        return f'TokenEmbedding({self.embedding.vocab_sz}, {self.embedding.embed_sz})'

class LearnedPositionalEmbedding:
    """Learned absolute position embeddings (the GPT-2 design).

    Weight shape: (max_len, d_model).

    __call__:
        x:      (B, T, d_model)   float token embeddings
        return: (B, T, d_model)   float, x + pos_emb after dropout
        Requires T <= max_len.
    """
    def __init__(self, max_len: int, d_model: int, dropout: float):
        self.embedding = nn.Embedding(max_len, d_model)
        self.dropout = dropout

    def __call__(self, x: Tensor) -> Tensor:
        # x: (B, T, d_model) -> same shape
        # Build a (B, T) tensor of positions [0..T-1], embed it, add to x,
        # then apply dropout.
        pos = Tensor.arange(0, x.shape[1], dtype=dtypes.int)
        embedded_positions = self.embedding(pos) #(T, d_model)
        return (x + embedded_positions).dropout(self.dropout)
        
        # raise NotImplementedError("Step 1: implement positional embedding")


In [81]:
def scaled_dot_product_attention(
    q: Tensor, k: Tensor, v: Tensor, mask: Optional[Tensor] = None,
    dropout: float = 0.0,
) -> Tensor:
    """Scaled dot-product attention.

    Inputs:
        q:    (B, h, T_q, d_k)
        k:    (B, h, T_k, d_k)
        v:    (B, h, T_k, d_v)            usually d_v == d_k
        mask: broadcastable to (B, h, T_q, T_k), additive (0 keep, -inf block).
              Often passed as (1, 1, T, T).

    Returns:
        (B, h, T_q, d_v)
    """
    scores = q @ k.transpose(-2,-1)/np.sqrt(cfg.d_model) +mask
    print(scores.shape, mask.shape)
    attn_wei = scores.softmax(axis=-1)
    attn_wei = attn_wei.dropout(dropout)
    return attn_wei @ v

    # scores = q @ k^T / sqrt(d_k); add mask if given; softmax over the last
    # axis; dropout the attention weights; then weight v by them.
    # raise NotImplementedError("Step 2: implement scaled dot-product attention")


In [88]:
class MultiHeadAttention:
    """Self-attention only, always causal.

    Linear weights w_q / w_k / w_v / w_o each have shape (d_model, d_model).
    w_o is a residual-stream write -- gets a smaller init in _init_weights.

    __call__:
        x:    (B, T, d_model)
        mask: broadcastable to (B, n_heads, T, T), additive (0 keep, -inf block);
              typically (1, 1, T, T) from make_causal_mask.
        return: (B, T, d_model)

    Internal heads run on (B, n_heads, T, d_k) where d_k = d_model // n_heads.
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.dropout = dropout

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

    def __call__(self, x: Tensor, mask: Optional[Tensor] = None) -> Tensor:
        # Project to Q, K, V; reshape to (B, n_heads, T, d_k); call
        # scaled_dot_product_attention; recombine heads back to (B, T, d_model);
        # project out with w_o.
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        q = q.reshape(cfg.batch_size, cfg.max_seq_len, self.n_heads, self.d_k).transpose(1,2)
        k = k.reshape(cfg.batch_size, cfg.max_seq_len, self.n_heads, self.d_k).transpose(1,2)
        v = v.reshape(cfg.batch_size, cfg.max_seq_len, self.n_heads, self.d_k).transpose(1,2)

        
        w_out = scaled_dot_product_attention(q, k, v, mask)
        print('w_out', w_out.shape)

        # w_out (2, 2, 6, 4)
        w_out = w_out.reshape(cfg.batch_size, cfg.max_seq_len, cfg.d_model)

        return self.w_o(w_out)
        # raise NotImplementedError("Step 2: implement multi-head attention")



In [83]:
class PositionwiseFeedForward:
    """Two linear transforms with GELU in between.

    tinygrad's .gelu() is the tanh approximation, matching GPT-2.

    Weights:
        w_1: (d_model, d_ff)   bias (d_ff,)
        w_2: (d_ff, d_model)   bias (d_model,)
    w_2 is a residual-stream write -- gets scaled init.

    __call__:
        x:      (B, T, d_model)
        return: (B, T, d_model)
        Internal hidden activation has shape (B, T, d_ff).
    """
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        self.w_1 = nn.Linear(d_model, d_ff, bias=True)
        self.w_2 = nn.Linear(d_ff, d_model, bias=True)
        self.dropout = dropout

    def __call__(self, x: Tensor) -> Tensor:
        # w_1 -> gelu -> dropout -> w_2.
        x = self.w_1(x)
        x = x.gelu()
        x = x.dropout(self.dropout)
        x = self.w_2(x)
        return x
    

In [84]:
class DecoderBlock:
    """One GPT-2 block: pre-norm self-attention + pre-norm FFN, residuals on top.

    Pre-norm:
        x = x + sublayer(LayerNorm(x))
    (vs GPT-1's post-norm: x = LayerNorm(x + sublayer(x)).)

    The residual stream is now the unnormalised highway running end-to-end.
    Each sublayer reads a normalised view of it and writes back into the raw
    stream.

    __call__:
        x:           (B, T, d_model)
        causal_mask: broadcastable to (B, n_heads, T, T); typically (1, 1, T, T).
        return:      (B, T, d_model)
    """
    def __init__(self, cfg: GPT2Config):
        self.norm_1 = nn.LayerNorm(cfg.d_model)
        self.self_attention = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.norm_2 = nn.LayerNorm(cfg.d_model)
        self.ffn = PositionwiseFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.dropout = cfg.dropout

    def __call__(self, x: Tensor, causal_mask: Tensor) -> Tensor:
        x = x + self.self_attention(self.norm_1(x), causal_mask).dropout(self.dropout)
        x = self.norm_2(x)
        x = x + self.ffn(x).dropout(self.dropout)
        # Pre-norm attention residual, then pre-norm FFN residual.
        # Apply self.dropout to each sublayer output before adding.
        # raise NotImplementedError("Step 3: implement the pre-norm decoder block")
        return x

In [85]:
def _init_weights(model: "GPT2") -> None:
    """GPT-2 init scheme.

    All Linear / Embedding weights ~ N(0, 0.02), biases zero, LayerNorm
    gamma=1 / beta=0 (which is tinygrad's default, so we leave it alone).

    The residual-write projections -- w_o in each attention block and w_2
    in each FFN -- are re-initialised with std = 0.02 / sqrt(2 * n_layers).
    Each layer makes two writes to the residual stream; without this
    down-scaling, activation variance grows linearly with depth.
    """
    std = 0.02
    n_layers = model.cfg.n_layers

    # Token + position embeddings
    model.token_embed.embedding.weight.assign(
        Tensor.normal(*model.token_embed.embedding.weight.shape, mean=0.0, std=std)
    )
    model.pos_embed.embedding.weight.assign(
        Tensor.normal(*model.pos_embed.embedding.weight.shape, mean=0.0, std=std)
    )

    # Per-block linears
    for block in model.blocks:
        for lin in (block.self_attention.w_q,
                    block.self_attention.w_k,
                    block.self_attention.w_v,
                    block.ffn.w_1):
            lin.weight.assign(Tensor.normal(*lin.weight.shape, mean=0.0, std=std))
            if lin.bias is not None:
                lin.bias.assign(Tensor.zeros(*lin.bias.shape))

        # Residual-write projections: scaled std
        scaled_std = std / math.sqrt(2 * n_layers)
        for lin in (block.self_attention.w_o, block.ffn.w_2):
            lin.weight.assign(Tensor.normal(*lin.weight.shape, mean=0.0, std=scaled_std))
            if lin.bias is not None:
                lin.bias.assign(Tensor.zeros(*lin.bias.shape))


In [86]:
class GPT2:
    """Decoder-only Transformer language model.

    Forward pass:
        tokens -> embed -> + pos -> N pre-norm blocks -> final LayerNorm
               -> tie-weighted projection to vocab logits.

    The output projection is tied to the input embedding -- there is no
    separate out_proj parameter. Use token_embed.weight.transpose() to
    project final hidden states back to vocab.

    Forward I/O:
        tokens:      (B, T)            int32 token ids in [0, vocab_size).
                                       Requires T <= cfg.max_seq_len.
        causal_mask: broadcastable to (B, n_heads, T, T), additive
                     (0 keep, -inf block); typically (1, 1, T, T).
        return:      (B, T, vocab_size)  float logits (pre-softmax).
    """
    def __init__(self, cfg: GPT2Config):
        self.cfg = cfg
        self.token_embed = TokenEmbedding(cfg.vocab_size, cfg.d_model)
        self.pos_embed = LearnedPositionalEmbedding(cfg.max_seq_len, cfg.d_model, cfg.dropout)
        self.blocks = [DecoderBlock(cfg) for _ in range(cfg.n_layers)]
        self.ln_f = nn.LayerNorm(cfg.d_model)

        _init_weights(self)

    def forward(self, tokens: Tensor, causal_mask: Tensor) -> Tensor:
        """Run a forward pass.

        tokens:      (B, T) int32 in [0, vocab_size).
        causal_mask: broadcastable to (B, n_heads, T, T); typically (1, 1, T, T).
        return:      (B, T, vocab_size) float logits.
        """
        # Embed tokens, add positions, run through self.blocks, apply ln_f,
        # project to vocab using the tied embedding weight.

        x = self.pos_embed(self.token_embed(tokens))
        
        for block in self.blocks:
            x = block(x, causal_mask)
        x = self.ln_f(x)
        
        return  x @ self.token_embed.embedding.weight.transpose()
        
        

        # raise NotImplementedError("Step 4: implement the forward pass")

    def __call__(self, tokens: Tensor, causal_mask: Tensor) -> Tensor:
        # raise NotImplementedError("Step 4: delegate to self.forward")
        return self.forward(tokens, causal_mask)


In [91]:
def lm_loss(logits: Tensor, target: Tensor, pad_idx: int) -> Tensor:
    """Cross-entropy for next-token prediction. No label smoothing.

    Args:
        logits:  (B, T, V) float, pre-softmax.
        target:  (B, T)    int  token ids in [0, V).
        pad_idx: int, positions where target == pad_idx are excluded from the mean.

    Returns:
        Scalar Tensor (shape ()) -- mean NLL over non-pad positions.
    """
    # log_softmax along the vocab axis, gather the target log-probs (one_hot
    # works fine here), mask out positions where target == pad_idx, return
    # the mean over the remaining positions.
    raise NotImplementedError("Step 5: implement language-modelling cross-entropy")


In [92]:
def train_step(model: GPT2, optimizer: AdamW, batch) -> float:
    """One fwd/bwd/step pass on a single batch.

    batch: (tokens_in, tokens_out, causal_mask).
    Returns the scalar loss value as a Python float.
    """
    tokens_in, tokens_out, causal_mask = batch

    logits = model(tokens_in, causal_mask)
    loss = lm_loss(logits, tokens_out, model.cfg.pad_idx)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.numpy().item()



In [93]:

class CosineScheduleWithWarmup:
    """Linear warmup -> cosine anneal to 0.

    lr(step):
        step:   int   current training step (0-indexed).
        return: float learning rate for that step.
    """
    def __init__(self, max_lr: float, warmup_steps: int, total_steps: int):
        self.max_lr = max_lr
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps

    def lr(self, step: int) -> float:
        if step < self.warmup_steps:
            return self.max_lr * step / max(1, self.warmup_steps)
        progress = (step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
        progress = min(progress, 1.0)
        return self.max_lr * 0.5 * (1.0 + math.cos(math.pi * progress))


def make_optimizer(model: GPT2, lr: float = 2.5e-4, weight_decay: float = 0.01):
    """AdamW over all parameters. Weight decay is applied uniformly here;
    the standard refinement is to split into decay / no-decay groups,
    excluding biases and LayerNorm gains.
    """
    params = get_parameters(model)
    return AdamW(params, lr=lr, b1=0.9, b2=0.999, eps=1e-8, weight_decay=weight_decay)



In [94]:
def train(cfg: GPT2Config, dataset, n_steps: int, max_lr: float = 2.5e-4) -> "GPT2":
    """Build a fresh GPT2 from cfg and train for up to n_steps batches.

    dataset: any iterable yielding (tokens_in, tokens_out, causal_mask) tuples.
    Returns the trained model.
    """
    model = GPT2(cfg)
    schedule = CosineScheduleWithWarmup(
        max_lr=max_lr,
        warmup_steps=min(200, max(1, n_steps // 10)),
        total_steps=n_steps,
    )
    optimizer = make_optimizer(model, lr=0.0)

    step = 0
    with Tensor.train():
        for batch in dataset:
            if step >= n_steps:
                break

            optimizer.lr = schedule.lr(step)
            loss = train_step(model, optimizer, batch)

            if step % 50 == 0:
                print(f"  step {step:4d}  loss {loss:.4f}  lr {optimizer.lr:.2e}")

            step += 1

    return model


def greedy_generate(
    model: GPT2, prompt: Tensor, max_new_tokens: int,
) -> Tensor:
    """Naive autoregressive generation -- no KV cache.

    Args:
        prompt:         (B, T_prompt) int32 token ids.
        max_new_tokens: number of tokens to append (one per forward pass).

    Returns:
        (B, T_prompt + max_new_tokens) int32. Requires
        T_prompt + max_new_tokens <= cfg.max_seq_len.

    Stretch goal: cache K, V from earlier steps so the per-step forward is
    O(T) rather than O(T^2).
    """
    # Loop max_new_tokens times. Each step:
    #   - rebuild causal_mask for the current length
    #   - forward pass to get logits (B, T, V)
    #   - take logits at the final position, argmax over vocab
    #   - concatenate the new token onto `tokens` along dim=1
    raise NotImplementedError("Step 6: implement greedy autoregressive generation")


In [95]:

model = GPT2(cfg)
B, T = 2, 6

tokens = Tensor(np.random.randint(1, cfg.vocab_size, (B, T)), dtype=dtypes.int32)
causal_mask = make_causal_mask(T)

logits = model(tokens, causal_mask)
print(f"  logits.shape = {logits.shape}  (expected ({B}, {T}, {cfg.vocab_size}))")
assert logits.shape == (B, T, cfg.vocab_size), f"shape mismatch: {logits.shape}"

n_params_total = len(get_parameters(model))
print(f"  total parameter tensors: {n_params_total}")
print("  (no separate out_proj.weight -- output projection is token_embed.weight^T)")
print("  PASSED\n")

# ------------------------------------------------------------------
# 2. Overfit on toy LM task -- loss should fall
# ------------------------------------------------------------------
print("=== 2. LM overfit (300 steps) ===")
dataset = ToyLMDataset(vocab_size=20, seq_len=8, n_batches=400, batch_size=32)
trained_model = train(cfg, dataset, n_steps=300)
print("  Training done\n")

# ------------------------------------------------------------------
# 3. Greedy generate from a short prompt
# ------------------------------------------------------------------
print("=== 3. Greedy generation ===")
prompt_arr = np.array([[3, 5, 7]])
prompt = Tensor(prompt_arr, dtype=dtypes.int32)
generated = greedy_generate(trained_model, prompt, max_new_tokens=7)
print(f"  prompt:    {prompt_arr[0].tolist()}")
print(f"  generated: {generated.numpy()[0].tolist()}")


(2, 2, 6, 6) (1, 1, 6, 6)
w_out (2, 2, 6, 4)
(2, 2, 6, 6) (1, 1, 6, 6)
w_out (2, 2, 6, 4)
  logits.shape = (2, 6, 100)  (expected (2, 6, 100))
  total parameter tensors: 38
  (no separate out_proj.weight -- output projection is token_embed.weight^T)
  PASSED

=== 2. LM overfit (300 steps) ===


ValueError: size mismatch, can't reshape ((32, 8, 8)) -> ((2, 6, 2, 4))